# Modulo 8 · Lezione 6 — Regolarizzazione: Dropout, Batch Norm, L1/L2

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Tecnica | A cosa serve? |
|---|---------|------------------|
| 1 | Il problema: overfitting | Diagnosticare quando la rete memorizza |
| 2 | L1 e L2 | Penalizzare pesi grandi |
| 3 | Dropout | Spegnere neuroni casualmente |
| 4 | Batch Normalization | Stabilizzare le attivazioni per strato |
| 5 | Early Stopping | Fermarsi quando smette di migliorare |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow: {tf.__version__}")

---
## 1. Il problema: overfitting

In [ ]:
# Provocare l'overfitting deliberatamente: pochi dati, rete grande
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.mnist.load_data()

# Solo 1000 campioni per addestrare → facilita l'overfitting
N_TRAIN = 1000
X_tr_m = X_tr_raw[:N_TRAIN].reshape(-1, 784).astype('float32') / 255.0
y_tr_m = y_tr_raw[:N_TRAIN]
X_te_m = X_te_raw.reshape(-1, 784).astype('float32') / 255.0
y_te_m = y_te_raw

print(f"Train piccolo: {X_tr_m.shape}  Test: {X_te_m.shape}")

def crear_modelo_base(dropout_rate=0.0, l2=0.0, batch_norm=False):
    """Crea il modello con diverse combinazioni di regolarizzazione."""
    reg = regularizers.l2(l2) if l2 > 0 else None
    capas = [layers.Input(shape=(784,))]
    for n in [512, 256, 128]:
        capas.append(layers.Dense(n, activation='relu', kernel_regularizer=reg))
        if batch_norm:
            capas.append(layers.BatchNormalization())
        if dropout_rate > 0:
            capas.append(layers.Dropout(dropout_rate))
    capas.append(layers.Dense(10, activation='softmax'))

    m = keras.Sequential(capas)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
    return m

# Modello senza regolarizzazione (riferimento)
tf.random.set_seed(42)
m_base = crear_modelo_base()
h_base = m_base.fit(X_tr_m, y_tr_m, epochs=50, batch_size=64,
                    validation_split=0.15, verbose=0)
_, acc_base_te = m_base.evaluate(X_te_m, y_te_m, verbose=0)

acc_base_tr  = h_base.history['accuracy'][-1]
acc_base_val = h_base.history['val_accuracy'][-1]
print(f"Senza regolarizzazione → Train: {acc_base_tr:.4f}, Val: {acc_base_val:.4f}, Test: {acc_base_te:.4f}")
print(f"Gap train-val: {acc_base_tr - acc_base_val:.4f}  ← overfitting")

---
## 2. L1 e L2: penalizzare pesi grandi

```
Senza regolarizzazione:  Loss = L(y, ŷ)
Con L2 (Ridge):          Loss = L(y, ŷ) + λ·Σwᵢ²     ← penalizza pesi grandi
Con L1 (Lasso):          Loss = L(y, ŷ) + λ·Σ|wᵢ|    ← favorisce pesi = 0 (sparse)
Con L1+L2 (ElasticNet): combina entrambe

λ (lambda): forza della regolarizzazione
  λ molto piccolo → poca regolarizzazione
  λ molto grande  → pesi → 0, underfitting
  Tipico: 1e-4 – 1e-2

In Keras:
  layers.Dense(n, kernel_regularizer=regularizers.l2(0.001))
  layers.Dense(n, kernel_regularizer=regularizers.l1(0.001))
  layers.Dense(n, kernel_regularizer=regularizers.l1_l2(l1=0.001, l2=0.001))
```

In [ ]:
# Confrontare L2 con diversi valori di λ
resultados_l2 = {'base': {'hist': h_base, 'acc_te': acc_base_te}}

for lam in [0.0001, 0.001, 0.01]:
    tf.random.set_seed(42)
    m = crear_modelo_base(l2=lam)
    h = m.fit(X_tr_m, y_tr_m, epochs=50, batch_size=64,
              validation_split=0.15, verbose=0)
    _, acc_te = m.evaluate(X_te_m, y_te_m, verbose=0)
    resultados_l2[f'L2={lam}'] = {'hist': h, 'acc_te': acc_te}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colores = ['gray', 'steelblue', 'darkorange', 'green']
for (nombre, res), color in zip(resultados_l2.items(), colores):
    acc_tr  = res['hist'].history['accuracy'][-1]
    acc_val = res['hist'].history['val_accuracy'][-1]
    label = f"{nombre} (tr={acc_tr:.3f}, val={acc_val:.3f}, te={res['acc_te']:.3f})"
    axes[0].plot(res['hist'].history['val_loss'],     color=color, lw=2, label=nombre)
    axes[1].plot(res['hist'].history['val_accuracy'], color=color, lw=2, label=label)

for ax in axes:
    ax.set_xlabel('Epoca'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
axes[0].set_title('Val Loss con L2'); axes[1].set_title('Val Accuracy con L2')
plt.tight_layout(); plt.show()

---
## 3. Dropout: spegnere neuroni casualmente

```
Durante l'addestramento, ogni neurone si 'spegne' con probabilità p (dropout_rate).
I neuroni attivi vengono scalati ×(1/(1-p)) per compensare.

Durante l'inferenza (predict/evaluate), si usano TUTTI i neuroni senza casualità.

Effetto:
  La rete impara rappresentazioni ridondanti e robuste.
  Equivale ad addestrare un 'ensemble' di 2^n sottoreti diverse.
  Molto efficace per evitare la coadattamento tra neuroni.

Rate tipici:
  0.2 – 0.3  per gli strati di ingresso
  0.4 – 0.5  per gli strati nascosti intermedi
  >0.6       raramente utile

In Keras:
  layers.Dropout(0.4)   # va dopo lo strato Dense, prima del successivo
```

In [ ]:
# Visualizzare cosa fa il Dropout
np.random.seed(0)
n_neuronas = 20
activaciones = np.random.rand(n_neuronas)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))

# Senza dropout
axes[0].bar(range(n_neuronas), activaciones, color='steelblue')
axes[0].set_title('Senza Dropout (tutti attivi)')

# Con dropout rate=0.5
mask = np.random.rand(n_neuronas) > 0.5
act_drop = activaciones * mask / 0.5  # scalare
colores_d = ['steelblue' if m else 'lightgray' for m in mask]
axes[1].bar(range(n_neuronas), act_drop, color=colores_d)
axes[1].set_title(f'Con Dropout(0.5) — {mask.sum()} attivi, {(~mask).sum()} spenti')

# Passo diverso (maschera casuale diversa)
mask2 = np.random.rand(n_neuronas) > 0.5
act_drop2 = activaciones * mask2 / 0.5
colores_d2 = ['steelblue' if m else 'lightgray' for m in mask2]
axes[2].bar(range(n_neuronas), act_drop2, color=colores_d2)
axes[2].set_title(f'Passo successivo — {mask2.sum()} attivi (maschera diversa)')

for ax in axes:
    ax.set_xlabel('Neurone'); ax.set_ylabel('Attivazione')
plt.tight_layout(); plt.show()
print("→ Ogni mini-batch usa una sottorete diversa → ensemble implicito")

In [ ]:
# Confrontare diversi tassi di Dropout
resultados_drop = {}
for rate in [0.0, 0.2, 0.4, 0.6]:
    tf.random.set_seed(42)
    m = crear_modelo_base(dropout_rate=rate)
    h = m.fit(X_tr_m, y_tr_m, epochs=60, batch_size=64,
              validation_split=0.15, verbose=0)
    _, acc_te = m.evaluate(X_te_m, y_te_m, verbose=0)
    acc_tr  = h.history['accuracy'][-1]
    acc_val = h.history['val_accuracy'][-1]
    resultados_drop[f'drop={rate}'] = {'hist': h, 'acc_te': acc_te,
                                        'acc_tr': acc_tr, 'acc_val': acc_val}
    print(f"Dropout={rate}: tr={acc_tr:.3f}, val={acc_val:.3f}, te={acc_te:.3f}, gap={acc_tr-acc_val:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colores = ['gray', 'steelblue', 'darkorange', 'green']
for (nombre, res), color in zip(resultados_drop.items(), colores):
    axes[0].plot(res['hist'].history['val_loss'],     color=color, lw=2, label=nombre)
    axes[1].plot(res['hist'].history['val_accuracy'], color=color, lw=2, label=nombre)
for ax in axes:
    ax.set_xlabel('Epoca'); ax.legend(); ax.grid(True, alpha=0.3)
axes[0].set_title('Val Loss'); axes[1].set_title('Val Accuracy')
plt.tight_layout(); plt.show()

---
## 4. Batch Normalization: stabilizzare le attivazioni

```
Problema: le attivazioni di ogni strato possono avere distribuzioni molto diverse
          e cambiare molto durante l'addestramento (internal covariate shift).

Batch Normalization (Ioffe & Szegedy, 2015):
  Per ogni mini-batch, normalizza l'uscita dello strato precedente:
    μ = mean(z)   σ² = var(z)
    z_norm = (z - μ) / sqrt(σ² + ε)
    out = γ · z_norm + β    ← γ e β sono parametri apprendibili

  Durante l'inferenza usa media/varianza mobili calcolate durante l'addestramento.

Vantaggi:
  Permette learning rate più alti → addestramento più veloce
  Riduce la dipendenza dall'inizializzazione
  Ha un leggero effetto regolarizzante (non sostituisce il Dropout)

Posizione tipica: Dense → BatchNorm → Activation (in alcune architetture)
              o: Dense(activation='relu') → BatchNorm  (la più comune in Keras)

In Keras:
  layers.BatchNormalization()
```

In [ ]:
# Visualizzare cosa fa BatchNorm sulle attivazioni
np.random.seed(42)
activaciones_sin_bn = np.random.exponential(scale=5, size=(1000,)) - 2  # distribuzione asimmetrica

# BN normalizza
mu  = activaciones_sin_bn.mean()
std = activaciones_sin_bn.std() + 1e-8
activaciones_con_bn = (activaciones_sin_bn - mu) / std

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(activaciones_sin_bn, bins=50, color='coral', edgecolor='white')
axes[0].set_title(f'Senza BatchNorm: μ={mu:.2f}, σ={std:.2f}')
axes[0].axvline(0, color='black', ls='--')
axes[0].set_xlabel('Attivazione')

axes[1].hist(activaciones_con_bn, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title(f'Con BatchNorm: μ≈{activaciones_con_bn.mean():.2f}, σ≈{activaciones_con_bn.std():.2f}')
axes[1].axvline(0, color='black', ls='--')
axes[1].set_xlabel('Attivazione normalizzata')

plt.suptitle('Effetto della Batch Normalization', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Confrontare: senza BN, con BN, con BN+Dropout
configs = [
    ('Senza regolarizzazione', dict(dropout_rate=0.0, l2=0.0, batch_norm=False)),
    ('BatchNorm',              dict(dropout_rate=0.0, l2=0.0, batch_norm=True)),
    ('Dropout(0.4)',           dict(dropout_rate=0.4, l2=0.0, batch_norm=False)),
    ('BN + Dropout(0.3)',      dict(dropout_rate=0.3, l2=0.0, batch_norm=True)),
]

print(f"{'Config':25} {'Train':>8} {'Val':>8} {'Test':>8} {'Gap':>8}")
print('-' * 60)
historiales = {}
for nombre, kwargs in configs:
    tf.random.set_seed(42)
    m = crear_modelo_base(**kwargs)
    h = m.fit(X_tr_m, y_tr_m, epochs=50, batch_size=64,
              validation_split=0.15, verbose=0)
    _, acc_te = m.evaluate(X_te_m, y_te_m, verbose=0)
    acc_tr  = h.history['accuracy'][-1]
    acc_val = h.history['val_accuracy'][-1]
    historiales[nombre] = h
    print(f"{nombre:25} {acc_tr:8.4f} {acc_val:8.4f} {acc_te:8.4f} {acc_tr-acc_val:8.4f}")

---
## 5. Early Stopping: fermarsi al momento ottimale

In [ ]:
# Visualizzare quando il modello 'inizia a memorizzare'
tf.random.set_seed(42)
m_es = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
m_es.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

# Addestrare senza EarlyStopping finché si vede l'overfitting
h_sin_es = m_es.fit(X_tr_m, y_tr_m, epochs=100, batch_size=64,
                    validation_split=0.15, verbose=0)

val_loss = np.array(h_sin_es.history['val_loss'])
mejor_epoca = val_loss.argmin()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(h_sin_es.history['loss'],     'steelblue', lw=2, label='Train')
axes[0].plot(h_sin_es.history['val_loss'], 'coral',     lw=2, label='Val')
axes[0].axvline(mejor_epoca, color='green', ls='--', lw=2,
                label=f'Epoca migliore ({mejor_epoca+1})')
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Loss')
axes[0].set_title('Senza Early Stopping'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(h_sin_es.history['accuracy'],     'steelblue', lw=2, label='Train')
axes[1].plot(h_sin_es.history['val_accuracy'], 'coral',     lw=2, label='Val')
axes[1].axvline(mejor_epoca, color='green', ls='--', lw=2, label=f'Fermarsi qui')
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Senza Early Stopping'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Val loss minimo all'epoca {mejor_epoca+1} su {len(val_loss)}")
print(f"→ EarlyStopping(patience=10) si sarebbe fermato ~all'epoca {mejor_epoca+11}")

In [ ]:
# Strategia combinata: la migliore pratica
tf.random.set_seed(42)
modelo_final = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(5e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(5e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax'),
])

modelo_final.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

h_final = modelo_final.fit(
    X_tr_m, y_tr_m,
    epochs=200, batch_size=64,
    validation_split=0.15,
    callbacks=[
        keras.callbacks.EarlyStopping('val_loss', patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=8, min_lr=1e-6),
    ],
    verbose=0
)

_, acc_te_final = modelo_final.evaluate(X_te_m, y_te_m, verbose=0)
acc_tr_final  = h_final.history['accuracy'][-1]
acc_val_final = h_final.history['val_accuracy'][-1]
n_epocas = len(h_final.history['loss'])

print(f"Con L2+BatchNorm+Dropout+EarlyStopping ({n_epocas} epoche):")
print(f"  Train: {acc_tr_final:.4f}  Val: {acc_val_final:.4f}  Test: {acc_te_final:.4f}")
print(f"  Gap train-val: {acc_tr_final-acc_val_final:.4f}")

---
## Pratica

### Esercizio 1 — Regolarizzazione su Fashion MNIST

**Consegna:** Fashion MNIST è più difficile di MNIST e più incline all'overfitting.

1. Usa solo 2000 campioni per l'addestramento
2. Confronta: senza regolarizzazione vs Dropout(0.4) vs L2(0.001)
3. Quale generalizza meglio sul test (tutti i 10000)?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
N = 2000
X_tr_n = X_tr[:N].reshape(-1, 784).astype('float32') / 255.0
y_tr_n = y_tr[:N]
X_te_n = X_te.reshape(-1, 784).astype('float32') / 255.0

def modelo_fashion(dropout=0.0, l2=0.0):
    reg = regularizers.l2(l2) if l2 > 0 else None
    capas = [layers.Input(shape=(784,))]
    for n in [256, 128]:
        capas.append(layers.Dense(n, activation='relu', kernel_regularizer=reg))
        if dropout > 0:
            capas.append(layers.Dropout(dropout))
    capas.append(layers.Dense(10, activation='softmax'))
    m = keras.Sequential(capas)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
    return m

configs = [
    ('Senza regolarizzazione', {}),
    ('Dropout(0.4)',           {'dropout': 0.4}),
    ('L2(0.001)',              {'l2': 0.001}),
]

print(f"{'Config':22} {'Train':>8} {'Val':>8} {'Test':>8}")
print('-' * 50)
for nombre, kwargs in configs:
    tf.random.set_seed(42)
    m = modelo_fashion(**kwargs)
    h = m.fit(X_tr_n, y_tr_n, epochs=60, batch_size=64,
              validation_split=0.15, verbose=0)
    _, acc_te = m.evaluate(X_te_n, y_te, verbose=0)
    acc_tr  = h.history['accuracy'][-1]
    acc_val = h.history['val_accuracy'][-1]
    print(f"{nombre:22} {acc_tr:8.4f} {acc_val:8.4f} {acc_te:8.4f}")
```
</details>

---
### Esercizio 2 — Dropout in test vs train

**Consegna:** Il Dropout si comporta in modo diverso in train e test. Verificalo:

1. Prendi un modello con Dropout(0.5) già addestrato
2. Usa `tf.keras.backend.learning_phase()` per attivare la modalità train
3. Fai 10 predizioni diverse sullo stesso campione con `training=True`
4. Osserva che le predizioni cambiano ogni volta (Monte Carlo Dropout)
5. Fai la media delle 10 predizioni → migliora rispetto a una singola predizione?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m.fit(X_tr, y_tr, epochs=10, batch_size=128, verbose=0)

# Prendere un'immagine
img = X_te[0:1]  # cifra 7
n_muestras = 20

# Monte Carlo Dropout: predizioni con dropout attivo
predicciones_mc = np.array([
    m(img, training=True).numpy()[0]  # training=True attiva il Dropout
    for _ in range(n_muestras)
])

pred_normal = m.predict(img, verbose=0)[0]  # training=False (senza dropout)
pred_mc     = predicciones_mc.mean(axis=0)  # media Monte Carlo

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = range(10)

# Varianza in ogni classe
axes[0].bar(x, predicciones_mc.mean(axis=0), yerr=predicciones_mc.std(axis=0),
            color='steelblue', capsize=5, label='Media ± std MC')
axes[0].bar(x, pred_normal, alpha=0.4, color='coral', label='Predizione normale')
axes[0].set_xticks(x); axes[0].set_xlabel('Classe (cifra)')
axes[0].set_title('Monte Carlo Dropout — incertezza per classe')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

# Variazione tra predizioni
for i, pred in enumerate(predicciones_mc):
    axes[1].plot(x, pred, 'o-', alpha=0.4, lw=0.8)
axes[1].plot(x, pred_mc, 'k-o', lw=2.5, markersize=7, label='Media MC')
axes[1].plot(x, pred_normal, 'r--s', lw=2, label='Senza dropout (normale)')
axes[1].set_xticks(x); axes[1].set_xlabel('Classe')
axes[1].set_title('20 predizioni MC diverse')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f"Predizione normale: classe {pred_normal.argmax()} ({pred_normal.max():.4f})")
print(f"Media MC ({n_muestras} campioni): classe {pred_mc.argmax()} ({pred_mc.max():.4f})")
print(f"Classe reale: {y_te[0]}")
print(f"\nIncertezza (std): {predicciones_mc.std(axis=0).round(4)}")
```
</details>

---
### Esercizio 3 — BatchNorm: velocità di convergenza

**Consegna:** BatchNorm permette di usare learning rate più alti.

1. Addestra due modelli identici (256 → 128 → 10) sull'intero MNIST:
   - Senza BatchNorm, lr=0.01
   - Con BatchNorm, lr=0.01
2. Confronta quale converge più velocemente nelle prime 10 epoche
3. Cosa succede senza BatchNorm con lr alto?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

configs = [
    ('Senza BN, lr=0.01',  False, 0.01),
    ('Con BN, lr=0.01',    True,  0.01),
    ('Senza BN, lr=0.001', False, 0.001),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colores = ['coral', 'steelblue', 'gray']

for (nombre, bn, lr), color in zip(configs, colores):
    tf.random.set_seed(42)
    capas = [layers.Input(shape=(784,))]
    for n in [256, 128]:
        capas.append(layers.Dense(n, activation='relu'))
        if bn:
            capas.append(layers.BatchNormalization())
    capas.append(layers.Dense(10, activation='softmax'))
    m = keras.Sequential(capas)
    m.compile(keras.optimizers.Adam(lr), 'sparse_categorical_crossentropy', ['accuracy'])
    h = m.fit(X_tr, y_tr, epochs=10, batch_size=128, validation_split=0.1, verbose=0)

    axes[0].plot(h.history['loss'],     color=color, lw=2, label=nombre)
    axes[1].plot(h.history['val_accuracy'], color=color, lw=2, label=nombre)

for ax in axes:
    ax.set_xlabel('Epoca'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
axes[0].set_title('Train Loss — prime 10 epoche')
axes[1].set_title('Val Accuracy — prime 10 epoche')
plt.tight_layout(); plt.show()

print("→ Con BatchNorm, lr=0.01 converge più velocemente rispetto a senza BN con lo stesso lr.")
print("  Senza BN e con lr alto, l'addestramento può essere instabile.")
```
</details>

---
### Esercizio 4 — Ricerca degli iperparametri di regolarizzazione

**Consegna:** Trova la combinazione ottimale:

1. Dataset: Fashion MNIST completo
2. Prova tutte le combinazioni di:
   - dropout_rate: [0.1, 0.3, 0.5]
   - l2: [0.0, 0.0005, 0.001]
3. Ogni modello: 3 epoche (veloce), valuta sul val
4. Mostra la classifica delle configurazioni per val_accuracy

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import numpy as np

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

dropout_rates = [0.1, 0.3, 0.5]
l2_vals      = [0.0, 0.0005, 0.001]

resultados = []
for dr in dropout_rates:
    for l2 in l2_vals:
        reg = regularizers.l2(l2) if l2 > 0 else None
        tf.random.set_seed(42)
        m = keras.Sequential([
            layers.Input(shape=(784,)),
            layers.Dense(256, activation='relu', kernel_regularizer=reg),
            layers.Dropout(dr),
            layers.Dense(128, activation='relu', kernel_regularizer=reg),
            layers.Dropout(dr),
            layers.Dense(10, activation='softmax'),
        ])
        m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
        h = m.fit(X_tr, y_tr, epochs=5, batch_size=128,
                  validation_split=0.1, verbose=0)
        val_acc = h.history['val_accuracy'][-1]
        resultados.append({'dropout': dr, 'l2': l2, 'val_acc': val_acc})
        print(f"dropout={dr}, l2={l2:.4f} → val_acc={val_acc:.4f}")

resultados.sort(key=lambda x: -x['val_acc'])
print("\n== Classifica ==")
for i, r in enumerate(resultados[:5], 1):
    print(f"{i}. dropout={r['dropout']}, l2={r['l2']:.4f} → {r['val_acc']:.4f}")
```
</details>

---
### Esercizio 5 (sfida) — Regolarizzazione personalizzata

**Consegna:** Implementa un callback personalizzato che monitori il gap train-val e attivi il Dropout dinamicamente:

1. Crea una classe `GapMonitor(keras.callbacks.Callback)` che registri `train_acc - val_acc` a ogni epoca
2. Al termine dell'addestramento, plotta il gap per epoca
3. Segnala a quale epoca il gap supera 0.05 (inizio dell'overfitting)
4. Usa questo callback nell'addestramento di Fashion MNIST con 5000 campioni

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

class GapMonitor(keras.callbacks.Callback):
    def __init__(self, umbral=0.05):
        super().__init__()
        self.umbral   = umbral
        self.gaps     = []
        self.alerta   = None

    def on_epoch_end(self, epoch, logs=None):
        tr  = logs.get('accuracy',     0)
        val = logs.get('val_accuracy', 0)
        gap = tr - val
        self.gaps.append(gap)
        if self.alerta is None and gap > self.umbral:
            self.alerta = epoch
            print(f"  [GapMonitor] Overfitting rilevato all'epoca {epoch+1}: gap={gap:.4f}")

    def plot(self):
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(self.gaps, 'steelblue', lw=2)
        ax.axhline(self.umbral, color='red', ls='--', label=f'Soglia ({self.umbral})')
        if self.alerta is not None:
            ax.axvline(self.alerta, color='orange', ls='--', label=f'Allerta epoca {self.alerta+1}')
        ax.set_xlabel('Epoca'); ax.set_ylabel('Gap (train - val accuracy)')
        ax.set_title('Monitoraggio del gap train-val'); ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr[:5000].reshape(-1, 784).astype('float32') / 255.0
y_tr = y_tr[:5000]
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

monitor = GapMonitor(umbral=0.05)
m.fit(X_tr, y_tr, epochs=60, batch_size=64, validation_split=0.15,
      callbacks=[monitor], verbose=0)

monitor.plot()
_, acc = m.evaluate(X_te, y_te, verbose=0)
print(f"Test accuracy: {acc:.4f}")
```
</details>

---
## Riepilogo

```python
from tensorflow.keras import regularizers

# L2 (weight decay)
layers.Dense(n, activation='relu', kernel_regularizer=regularizers.l2(0.001))

# Dropout: va dopo lo strato denso
layers.Dense(n, activation='relu')
layers.Dropout(0.4)   # spegne il 40% dei neuroni in ogni passo di train

# Batch Normalization: va dopo Dense, prima/dopo l'attivazione
layers.Dense(n, activation='relu')
layers.BatchNormalization()

# Early Stopping
keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

# Strategia consigliata (in ordine di priorità):
# 1. EarlyStopping sempre
# 2. Dropout se c'è overfitting evidente
# 3. BatchNorm se l'addestramento è lento o instabile
# 4. L2 come complemento, non come sostituto
```

| Tecnica | Quando usarla | Parametro chiave |
|---------|---------------|-----------------|
| L2 | Sempre disponibile, complemento leggero | λ ∈ [1e-4, 1e-2] |
| Dropout | Overfitting marcato, reti dense | rate ∈ [0.2, 0.5] |
| BatchNorm | Addestrare più velocemente, reti profonde | — (senza iperparametri critici) |
| EarlyStopping | Sempre | patience ∈ [5, 20] |

## Cosa viene nella prossima lezione?
Nella **Lezione 7** impareremo le **Reti Convoluzionali (CNN)** — l'architettura che ha rivoluzionato la visione artificiale, capace di rilevare pattern visivi direttamente nei pixel.